# Section 6 — Practical 2: Docker + Local Ollama

**Objective:** Package a real FastAPI agent into a Docker container, bind its API to `127.0.0.1:8768`, and connect it to Ollama running on your Windows host using `host.docker.internal`. **No AWS, no paid API.**

**Safety:** Use only fictional training text. This is a *local container deployment*, **not** a production-ready public cloud deployment. Keep Docker Desktop open. Do not copy PowerShell prompts into notebook cells.

**Expected:** eight code cells. Run strictly in order. Cell 7 makes a *local* real-model request and can take time on a CPU laptop. Cell 8 saves the report and prints clean-up commands.

In [ ]:
# Cell 1 — Windows Docker + Ollama readiness (no model invocation)
import shutil, subprocess, sys, urllib.request, json

def cmd(args, cwd=None, timeout=60):
    p = subprocess.run(args, cwd=cwd, capture_output=True, text=True, timeout=timeout, shell=False)
    if p.returncode:
        raise RuntimeError(f"Command failed: {' '.join(args)}\n{(p.stderr or p.stdout)[-1800:]}")
    return p.stdout.strip()

print('Python:', sys.version.split()[0])
assert shutil.which('docker'), 'Docker not found. Start/install Docker Desktop and restart JupyterLab.'
print('Docker:', cmd(['docker','--version']))
print('Docker daemon:', cmd(['docker','info','--format','{{.ServerVersion}}'], timeout=25))
with urllib.request.urlopen('http://127.0.0.1:11434/api/tags', timeout=8) as r:
    names = [m.get('name') for m in json.load(r).get('models', [])]
print('Ollama models:', names)
assert 'qwen3:1.7b' in names, 'In PowerShell run: ollama pull qwen3:1.7b'
print('PASS: Docker Desktop + local Ollama available')


## Prepare the container files

Cell 2 writes `app.py`, `Dockerfile`, `compose.yaml`, and `requirements.txt` under a new `section6_docker_training` folder. This makes the notebook independent of file uploads. You can inspect the files later.

In [ ]:
# Cell 2 — Write all application/deployment files (no network operations)
from pathlib import Path
PROJECT_DIR = Path.cwd() / 'section6_docker_training'
PROJECT_DIR.mkdir(exist_ok=True)
ASSETS = {'app.py': '"""Section 6 / Practical 2: local-only containerized training document analyst.\nFictional data only. No AWS services. The application is not public-facing.\n"""\nimport os\nimport time\nimport threading\nfrom urllib.request import Request, urlopen\nfrom urllib.error import HTTPError, URLError\nimport json\n\nfrom fastapi import FastAPI, HTTPException\nfrom pydantic import BaseModel, Field\n\nOLLAMA_URL = os.getenv("OLLAMA_BASE_URL", "http://host.docker.internal:11434").rstrip("/")\nMODEL = os.getenv("OLLAMA_MODEL", "qwen3:1.7b")\nSEMAPHORE = threading.BoundedSemaphore(1)\napp = FastAPI(title="Bootcamp Section 6 Docker AI Agent", version="1.0")\n\nclass AnalyzeRequest(BaseModel):\n    document_text: str = Field(min_length=20, max_length=5000)\n    dry_run: bool = False\n\n\ndef ollama_json(path, payload=None, timeout=8):\n    url = OLLAMA_URL + path\n    data = json.dumps(payload).encode("utf-8") if payload is not None else None\n    req = Request(url, data=data, headers={"Content-Type": "application/json"} if data else {},\n                  method="POST" if data else "GET")\n    with urlopen(req, timeout=timeout) as r:\n        return json.load(r)\n\n@app.get("/health")\ndef health():\n    return {"status": "ok", "service": "section6-docker-training", "scope": "local"}\n\n@app.get("/ready")\ndef ready():\n    try:\n        tags = ollama_json("/api/tags", timeout=7)\n        names = [m.get("name") for m in tags.get("models", [])]\n        return {"status": "ready" if MODEL in names else "model_missing", "model": MODEL}\n    except (HTTPError, URLError, OSError, ValueError, TimeoutError) as e:\n        raise HTTPException(503, detail=f"Ollama not reachable from container ({type(e).__name__})")\n\n@app.post("/analyze")\ndef analyze(req: AnalyzeRequest):\n    started = time.monotonic()\n    if req.dry_run:\n        return {"mode": "dry-run", "analysis": "Validation succeeded; AI not invoked.", "elapsed_ms": 0}\n    if not SEMAPHORE.acquire(blocking=False):\n        raise HTTPException(status_code=429, detail="Local model is busy; retry later")\n    try:\n        base = {"model": MODEL, "stream": False, "think": False,\n                "messages": [\n                    {"role": "system", "content": "Analyze the fictional training document in simple Hindi. State parties, term, charges, dates, concerns. Do not invent missing facts. This is not legal advice."},\n                    {"role": "user", "content": req.document_text},\n                ]}\n        # A smaller Qwen3 model can spend output budget on thinking if not controlled.\n        answer = ""\n        for tokens in (900, 1400):\n            payload = dict(base)\n            payload["options"] = {"num_predict": tokens, "temperature": 0.2}\n            try:\n                response = ollama_json("/api/chat", payload, timeout=210)\n            except (HTTPError, URLError, OSError, ValueError, TimeoutError) as e:\n                raise HTTPException(status_code=502, detail=f"Local Ollama request failed: {type(e).__name__}")\n            answer = (response.get("message") or {}).get("content", "").strip()\n            if answer:\n                break\n        if not answer:\n            raise HTTPException(status_code=502, detail="Ollama returned an empty final answer")\n        return {"mode": "docker-local-ollama", "model": MODEL, "analysis": answer,\n                "elapsed_ms": round((time.monotonic()-started)*1000, 1)}\n    finally:\n        SEMAPHORE.release()\n', 'requirements.txt': 'fastapi>=0.115,<1.0\nuvicorn>=0.32,<1.0\n', 'Dockerfile': 'FROM python:3.12-slim\nWORKDIR /app\nCOPY requirements.txt .\nRUN pip install --no-cache-dir -r requirements.txt && useradd --create-home --uid 10001 appuser\nCOPY app.py .\nUSER appuser\nEXPOSE 8000\nCMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000", "--no-access-log"]\n', 'compose.yaml': 'services:\n  ai-agent:\n    build: .\n    image: ai-bootcamp-section6:local\n    container_name: ai-bootcamp-section6-local\n    ports:\n      - "127.0.0.1:8768:8000"\n    environment:\n      OLLAMA_BASE_URL: "http://host.docker.internal:11434"\n      OLLAMA_MODEL: "qwen3:1.7b"\n    restart: "no"\n    read_only: true\n    tmpfs:\n      - /tmp\n    security_opt:\n      - no-new-privileges:true\n', '.dockerignore': '__pycache__/\n.ipynb_checkpoints/\n*.ipynb\n*.zip\n*.json\n*.md\n*.log\n'}
for filename, contents in ASSETS.items():
    (PROJECT_DIR / filename).write_text(contents,encoding='utf-8')
print('Project:', PROJECT_DIR)
print('Files:', sorted(p.name for p in PROJECT_DIR.iterdir()))
print('PASS: Local FastAPI and Docker deployment files created')


In [ ]:
# Cell 3 — Validate application source and container configuration
import ast
source = (PROJECT_DIR/'app.py').read_text(encoding='utf-8')
ast.parse(source)
compose_text=(PROJECT_DIR/'compose.yaml').read_text(encoding='utf-8')
assert '127.0.0.1:8768:8000' in compose_text, 'Local-only host port mapping missing'
assert 'host.docker.internal:11434' in compose_text, 'Ollama host address missing'
assert 'read_only: true' in compose_text
print('PASS: Python syntax and safe localhost-only Docker configuration valid')


In [ ]:
# Cell 4 — Build Docker image (first time downloads the Python base image)
output = cmd(['docker','compose','-f','compose.yaml','build'], cwd=str(PROJECT_DIR),timeout=900)
print('\n'.join(output.splitlines()[-15:]))
print('PASS: Docker image built successfully')


In [ ]:
# Cell 5 — Start container and wait for its health endpoint
import time, requests
output = cmd(['docker','compose','-f','compose.yaml','up','-d','--no-build'], cwd=str(PROJECT_DIR), timeout=90)
print(output[-800:])
LOCAL_API='http://127.0.0.1:8768'
for attempt in range(40):
    try:
        r=requests.get(LOCAL_API+'/health',timeout=2)
        if r.ok and r.json().get('service')=='section6-docker-training':
            break
    except requests.RequestException:
        pass
    time.sleep(0.5)
else:
    print(cmd(['docker','compose','-f','compose.yaml','logs','--tail','25'],cwd=str(PROJECT_DIR)))
    raise RuntimeError('Container did not become healthy; inspect Docker Desktop')
print('PASS: Container running on', LOCAL_API)
print('API documentation:', LOCAL_API+'/docs')


In [ ]:
# Cell 6 — Test HTTP endpoints, validation, local Ollama reachability
sample_document = ('यह केवल प्रशिक्षण के लिए काल्पनिक अनुबंध है। '
    'ABC Services, XYZ Retail को 12 महीनों के लिए आईटी सेवाएँ देगा। '
    'मासिक शुल्क ₹50,000 है। बिल के 45 दिनों के भीतर भुगतान किया जाएगा। '
    '30 दिन के लिखित नोटिस से अनुबंध समाप्त हो सकता है।')
health=requests.get(LOCAL_API+'/health',timeout=10)
assert health.status_code==200 and health.json()['status']=='ok'
bad=requests.post(LOCAL_API+'/analyze',json={'document_text':'short'},timeout=10)
assert bad.status_code==422, bad.text
dry=requests.post(LOCAL_API+'/analyze',json={'document_text':sample_document,'dry_run':True},timeout=10)
assert dry.status_code==200 and dry.json()['mode']=='dry-run',dry.text
ready=requests.get(LOCAL_API+'/ready',timeout=15)
print('Ollama readiness status:', ready.status_code,ready.text[:300])
assert ready.status_code==200 and ready.json()['status']=='ready', (
    'Container cannot reach Ollama on host.docker.internal:11434. '
    'Do not expose Ollama to your local network. Ask for targeted Windows Docker networking help.')
print('PASS: Docker API health, input validation, dry-run, Ollama readiness')


In [ ]:
# Cell 7 — Real AI request through Docker -> host Ollama (training data only)
response=requests.post(LOCAL_API+'/analyze',json={
    'document_text':sample_document, 'dry_run':False},timeout=290)
print('HTTP status:',response.status_code)
assert response.status_code==200,response.text[:1000]
live_report=response.json()
assert live_report['mode']=='docker-local-ollama' and live_report['analysis'].strip()
print(live_report['analysis'][:2500])
print('PASS: Real Qwen3 response received through Dockerized FastAPI')


In [ ]:
# Cell 8 — Save evidence/report and print optional cleanup command
from pathlib import Path
assert live_report['analysis'].strip(), 'Run Cell 7 first'
report_path=Path.cwd()/'section6_docker_report.json'
report_data={
 'section':6,'practical':'Local Docker container deployment (not AWS)',
 'image':'ai-bootcamp-section6:local', 'url':LOCAL_API,
 'endpoints':['/health','/ready','/analyze','/docs'],
 'training_sample':sample_document, 'result':live_report,
}
report_path.write_text(json.dumps(report_data,ensure_ascii=False,indent=2),encoding='utf-8')
assert report_path.exists() and report_path.stat().st_size>100
print('Saved:',report_path)
print('PASS: Section 6 Practical 2 — Local Docker + Ollama completed')
print('To stop the container later, run in PowerShell from',PROJECT_DIR)
print('docker compose down')
